# Figures: traffic simulation

Notebook for the simulation-model figures. It reads the available daily simulations in `article/data/uniq/`, compares uncorrected and corrected outputs, and saves figures in `article/results/figures/simulazione/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'article' / 'data' / 'uniq').exists():
            return candidate
    raise FileNotFoundError('Run the notebook inside the covid_hep repository.')

ROOT = find_root()
DATA = ROOT / 'article' / 'data' / 'uniq'
OUT = ROOT / 'article' / 'results' / 'figures' / 'simulazione'
OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='talk', font_scale=1.25)
plt.rcParams.update({'font.size': 18, 'axes.titlesize': 22, 'axes.labelsize': 19, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'legend.fontsize': 16, 'legend.title_fontsize': 16, 'pdf.fonttype': 42, 'ps.fonttype': 42})
COLORS = {2020: '#146C94', 2021: '#E15759'}

def save_figure(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(OUT / f'{name}.{ext}', dpi=300, bbox_inches='tight')
    plt.close(fig)

corrected_files = sorted(DATA.glob('*_uniq_*.csv.corrected'))
assert corrected_files, 'No corrected simulation output found.'
print(f'{len(corrected_files)} corrected simulations available; output: {OUT}')

In [ ]:
def load_day(path):
    frame = pd.read_csv(path, sep=';')
    hours = [c for c in frame.columns if c[:4] in {'2020', '2021'}]
    date = pd.Timestamp(hours[0]).normalize()
    grouped = frame.groupby('poly_cid')[hours].sum()
    active_links = grouped.index[grouped.sum(axis=1) > 0]
    hourly_links = (grouped.loc[active_links].T.rename_axis('time').reset_index()
                    .melt(id_vars='time', var_name='poly_cid', value_name='flow'))
    hourly_links['time'] = pd.to_datetime(hourly_links['time'])
    hourly_links['hour'] = hourly_links['time'].dt.hour
    hourly_links['date'] = date
    hourly_links['year'] = date.year
    daily_segments = grouped.sum(axis=1).rename('daily_flow').reset_index()
    daily_segments['date'] = date
    daily_segments['year'] = date.year
    return hourly_links, daily_segments

def read_collection(files):
    hourly, segment = zip(*(load_day(path) for path in files))
    return pd.concat(hourly, ignore_index=True), pd.concat(segment, ignore_index=True)

hourly_corrected, segment_corrected = read_collection(corrected_files)
def hourly_profile(hourly):
    return (hourly.groupby(['year', 'hour'])['flow']
            .agg(mean_flow='mean', median_flow='median',
                 q25_flow=lambda values: values.quantile(.25),
                 q75_flow=lambda values: values.quantile(.75))
            .reset_index())

typical_hour = hourly_profile(hourly_corrected)
segment_mean = segment_corrected.groupby(['year', 'poly_cid'], as_index=False)['daily_flow'].mean()
typical_hour.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.5), constrained_layout=True)
for year, group in typical_hour.groupby('year'):
    ax.plot(group['hour'], group['mean_flow'], marker='o', ms=4, lw=2, label=str(year), color=COLORS[year])
    ax.fill_between(group['hour'], group['q25_flow'], group['q75_flow'], alpha=.12, color=COLORS[year])
ax.set(title='Mean simulated hourly flow on active road links', xlabel='Hour of day', ylabel='Mean link crossings per hour')
ax.set_xticks(range(0, 24, 3))
ax.legend(title='Campaign')
save_figure(fig, 'simulated_hourly_profile')
typical_hour.to_csv(OUT / 'simulated_hourly_profile.csv', index=False)

In [ ]:
network = gpd.read_file(DATA / 'carto.geojson')
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), constrained_layout=True)
positive = segment_mean.loc[segment_mean['daily_flow'] > 0, 'daily_flow']
norm = LogNorm(vmin=max(1, positive.quantile(.05)), vmax=positive.quantile(.98))
for ax, year in zip(axes, sorted(segment_mean['year'].unique())):
    map_data = network.merge(segment_mean.query('year == @year'), on='poly_cid', how='left')
    map_data['daily_flow'] = map_data['daily_flow'].fillna(0)
    network.plot(ax=ax, color='#e8e8e8', linewidth=.25)
    map_data.query('daily_flow > 0').plot(column='daily_flow', ax=ax, cmap='viridis', norm=norm, linewidth=1.0)
    ax.set_title(f'Mean simulated daily load - {year}')
    ax.set_axis_off()
sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes, shrink=.70, label='Link crossings / day (log scale)')
save_figure(fig, 'simulated_network_load')

In [ ]:
raw_files = [Path(str(path).removesuffix('.corrected')) for path in corrected_files]
raw_files = [path for path in raw_files if path.exists()]
hourly_raw, _ = read_collection(raw_files)
raw_profile = hourly_profile(hourly_raw).assign(version='Uncorrected output')
corrected_profile = typical_hour.assign(version='Corrected output')
comparison = pd.concat([raw_profile, corrected_profile], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True, constrained_layout=True)
for ax, year in zip(axes, sorted(comparison['year'].unique())):
    subset = comparison.query('year == @year')
    sns.lineplot(data=subset, x='hour', y='mean_flow', hue='version', marker='o', ax=ax, palette=['#7F7F7F', COLORS[year]])
    ax.set(title=str(year), xlabel='Hour of day', ylabel='Mean link crossings per hour' if ax is axes[0] else '')
    ax.set_xticks(range(0, 24, 4))
    ax.legend(title='')
fig.suptitle('Effect of correction on simulated profiles', y=1.03)
save_figure(fig, 'simulation_correction_effect')
comparison.to_csv(OUT / 'simulation_correction_effect.csv', index=False)
print('Created figures:', sorted(p.name for p in OUT.glob('*.pdf')))